# 🔧 Lesson 08b — Intelligence Core: RAG + ML Hybrid Wiring
**Duration:** ~1 hour | Part of Capstone Preparation

In [ ]:
# ── Shared imports (run this first) ──────────────────────────────
import os, json, time, datetime, re
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator
from datasets import load_dataset
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

Path("data/capstone").mkdir(parents=True, exist_ok=True)
print("✅  Module 8 imports ready")

---
## 08b — Intelligence Core: RAG + ML Hybrid Wiring

**Goal:** Wire the ChromaDB policy store and XGBoost churn model together into `hybrid_route()`, producing a routing decision and rationale log.


In [ ]:
# ── Build synthetic policy documents + ChromaDB ───────────────────
import chromadb
from chromadb.utils import embedding_functions

# Synthetic bank policies
POLICY_TOPICS = [
    "credit card dispute resolution",
    "mortgage late payment fees",
    "account overdraft protection",
    "fraud reporting procedure",
    "personal loan early repayment",
    "savings account interest calculation",
    "wire transfer limits",
    "KYC identity verification",
    "GDPR data deletion rights",
    "anti-money laundering checks",
]

def generate_policy_doc(topic: str) -> str:
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=200,
        messages=[{"role":"user","content":
            f"Write a 150-word formal bank policy document about: {topic}. "
            "Start with 'BANK POLICY:' and use formal language."}]
    )
    return msg.choices[0].message.content.strip()

# Generate 30 policy docs (10 topics × 3 variations)
policy_docs = []
for topic in POLICY_TOPICS:
    for variant in range(3):
        doc = generate_policy_doc(f"{topic} (variant {variant+1})")
        policy_docs.append({"id": f"policy_{len(policy_docs):03d}", "text": doc, "topic": topic})

print(f"✅ Generated {len(policy_docs)} policy documents")

In [ ]:
# ── Index into ChromaDB ───────────────────────────────────────────
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("bank_policies")
except Exception:
    pass

collection = chroma_client.create_collection("bank_policies", embedding_function=ef)
collection.add(
    ids       = [p["id"]   for p in policy_docs],
    documents = [p["text"] for p in policy_docs],
    metadatas = [{"topic": p["topic"]} for p in policy_docs],
)
print(f"✅ Indexed {collection.count()} documents in ChromaDB")

def retrieve_policy(query: str, n_results: int = 2) -> str:
    """Retrieve top policy snippets for a query."""
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    return "\n---\n".join(docs)

# Quick test
snippet = retrieve_policy("credit card dispute")
print("\nSample retrieval:\n", snippet[:200])

In [ ]:
# ── XGBoost churn model (Telco proxy) ─────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import numpy as np

# Load Telco churn dataset
try:
    telco = pd.read_csv("data/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    print(f"Telco dataset: {telco.shape}")
except FileNotFoundError:
    # Generate a minimal synthetic dataset for testing
    print("⚠️  Telco CSV not found — generating synthetic data for demo")
    np.random.seed(42)
    n = 500
    telco = pd.DataFrame({
        "tenure":           np.random.randint(1, 72, n),
        "MonthlyCharges":   np.random.uniform(20, 120, n),
        "TotalCharges":     np.random.uniform(100, 8000, n),
        "Contract":         np.random.choice(["Month-to-month","One year","Two year"], n),
        "PaymentMethod":    np.random.choice(["Electronic check","Mailed check","Bank transfer"], n),
        "InternetService":  np.random.choice(["DSL","Fiber optic","No"], n),
        "Churn":            np.random.choice(["Yes","No"], n, p=[0.27, 0.73]),
    })

# Feature engineering
telco = telco.dropna()
telco["Churn_bin"] = (telco["Churn"] == "Yes").astype(int)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in ["Contract","PaymentMethod","InternetService"] if c in telco.columns]

X = telco[num_cols + cat_cols]
y = telco["Churn_bin"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
])

model = Pipeline([
    ("pre", preprocess),
    ("clf", xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42,
                               eval_metric="logloss", verbosity=0))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print(f"✅ XGBoost trained | AUC = {auc:.3f}")

In [ ]:
# ── Hybrid router ──────────────────────────────────────────────────
HYBRID_LOG = []

def hybrid_route(customer_features: dict, query: str) -> dict:
    """
    Route a customer query:
    - If ML confidence is HIGH (proba > 0.75 or < 0.25): use ML prediction
    - Otherwise: use LLM + RAG for nuanced assessment
    Returns a routing decision dict.
    """
    # Build feature row (use defaults for missing)
    defaults = {"tenure":12, "MonthlyCharges":60.0, "TotalCharges":720.0,
                "Contract":"Month-to-month","PaymentMethod":"Electronic check",
                "InternetService":"DSL"}
    row = {**defaults, **customer_features}
    X_row = pd.DataFrame([{k: row[k] for k in num_cols + cat_cols}])
    ml_proba = float(model.predict_proba(X_row)[0, 1])

    if ml_proba > 0.75 or ml_proba < 0.25:
        route = "ml"
        rationale = f"High-confidence ML prediction (proba={ml_proba:.2f})"
        final_prediction = "high_risk" if ml_proba > 0.75 else "low_risk"
    else:
        # Use RAG + LLM for uncertain cases
        policy_context = retrieve_policy(query)
        llm_resp = lm_client.chat.completions.create(
            model="local-model", max_tokens=150,
            messages=[{"role":"user","content":
                f"Given policy context: {policy_context[:400]}\n"
                f"And customer churn probability: {ml_proba:.2f}\n"
                f"Query: {query}\n"
                "Classify churn risk as high_risk or low_risk. Reply JSON: {risk, reason}"}]
        )
        try:
            llm_data = json.loads(
                llm_resp.choices[0].message.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            )
        except Exception:
            llm_data = {"risk": "medium_risk", "reason": "parse error"}
        route = "llm_rag"
        rationale = f"Uncertain ML (proba={ml_proba:.2f}); LLM+RAG says: {llm_data.get('reason','')}"
        final_prediction = llm_data.get("risk", "medium_risk")

    entry = {
        "customer_features": customer_features,
        "ml_proba": ml_proba,
        "route": route,
        "final_prediction": final_prediction,
        "rationale": rationale,
        "cost_usd": 0.0 if route == "ml" else 0.0002,  # LLM call cost estimate
        "ts": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    HYBRID_LOG.append(entry)
    return entry

# Test on 5 customers
for i in range(5):
    result = hybrid_route(
        {"tenure": i*12, "MonthlyCharges": 50 + i*15},
        "Will this customer churn?"
    )
    print(f"Customer {i}: proba={result['ml_proba']:.2f} | route={result['route']} | pred={result['final_prediction']}")

In [ ]:
# ── Save hybrid routing log ───────────────────────────────────────
log_path = Path("data/capstone/hybrid_routing_log.jsonl")
with open(log_path, "w") as f:
    for entry in HYBRID_LOG:
        f.write(json.dumps(entry) + "\n")
print(f"✅ Saved {len(HYBRID_LOG)} routing decisions → {log_path}")

check([
    (collection.count() >= 30,             "ChromaDB has ≥ 30 policy docs indexed"),
    (auc >= 0.68,                          f"XGBoost AUC ≥ 0.68 (got {auc:.3f})"),
    (len(HYBRID_LOG) >= 5,                 "Hybrid router ran on ≥ 5 customers"),
    (log_path.exists(),                    "hybrid_routing_log.jsonl saved"),
    (all("ml_proba" in e for e in HYBRID_LOG), "ml_proba in every log entry"),
], exercise_id="08b-1")

---
## ✅ Lesson 08b Readiness
- ☐ ChromaDB with ≥ 30 policies indexed
- ☐ XGBoost AUC ≥ 0.68
- ☐ `hybrid_routing_log.jsonl` saved with all required fields